In [9]:
from datasets import load_dataset

dataset = load_dataset("microsoft/ms_marco", "v1.1", split="train")

print(dataset)
print(dataset[0])

Dataset({
    features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
    num_rows: 82326
})
{'answers': ['Results-Based Accountability is a disciplined way of thinking and taking action that communities can use to improve the lives of children, youth, families, adults and the community as a whole.'], 'passages': {'is_selected': [0, 0, 0, 0, 0, 1, 0, 0, 0, 0], 'passage_text': ["Since 2007, the RBA's outstanding reputation has been affected by the 'Securency' or NPA scandal. These RBA subsidiaries were involved in bribing overseas officials so that Australia might win lucrative note-printing contracts. The assets of the bank include the gold and foreign exchange reserves of Australia, which is estimated to have a net worth of A$101 billion. Nearly 94% of the RBA's employees work at its headquarters in Sydney, New South Wales and at the Business Resumption Site.", "The Reserve Bank of Australia (RBA) came into being on 14 January 1960 as Australia 's ce

In [10]:
import json

# json.dumps formats a Python dictionary into neatly indented, readable text,
# instead of one long unreadable line. indent=2 means 2 spaces per nesting level.
print(json.dumps(dataset[0], indent=2))

{
  "answers": [
    "Results-Based Accountability is a disciplined way of thinking and taking action that communities can use to improve the lives of children, youth, families, adults and the community as a whole."
  ],
  "passages": {
    "is_selected": [
      0,
      0,
      0,
      0,
      0,
      1,
      0,
      0,
      0,
      0
    ],
    "passage_text": [
      "Since 2007, the RBA's outstanding reputation has been affected by the 'Securency' or NPA scandal. These RBA subsidiaries were involved in bribing overseas officials so that Australia might win lucrative note-printing contracts. The assets of the bank include the gold and foreign exchange reserves of Australia, which is estimated to have a net worth of A$101 billion. Nearly 94% of the RBA's employees work at its headquarters in Sydney, New South Wales and at the Business Resumption Site.",
      "The Reserve Bank of Australia (RBA) came into being on 14 January 1960 as Australia 's central bank and banknote iss

In [4]:
# We'll only use the first N queries from the dataset to keep things fast.
# You can increase this later once everything works, with zero code changes needed.
NUM_QUERIES = 3000

# Take a subset of the dataset — the first NUM_QUERIES rows.
# .select() is a Hugging Face Dataset method that returns a smaller Dataset
# containing only the rows at the given index positions.
subset = dataset.select(range(NUM_QUERIES))

# This dictionary will store our flattened corpus: {passage_id: passage_text}
corpus = {}

# This list will store, for each query, its text and which passage IDs are relevant.
queries_data = []

# A simple counter we'll use to assign a unique ID to every passage we encounter.
passage_id_counter = 0

# Loop through every query row in our subset.
for row in subset:
    query_text = row["query"]
    query_id = row["query_id"]

    # row["passages"] is a dictionary containing three PARALLEL lists:
    # passage_text[i], is_selected[i], and url[i] all describe the SAME i-th passage.
    passage_texts = row["passages"]["passage_text"]
    is_selected_flags = row["passages"]["is_selected"]

    # This will hold the passage IDs relevant to THIS specific query.
    relevant_ids_for_this_query = []

    # zip() lets us loop through two lists together, pairing up matching positions.
    # So we get (text_of_passage_0, flag_of_passage_0), (text_of_passage_1, flag_of_passage_1), etc.
    for passage_text, is_selected in zip(passage_texts, is_selected_flags):
        # Assign this passage a brand-new unique ID, then add it to our corpus.
        current_id = passage_id_counter
        corpus[current_id] = passage_text
        passage_id_counter += 1

        # If this passage was marked relevant (is_selected == 1), remember its ID
        # as a "correct answer" for this query — this becomes our ground truth.
        if is_selected == 1:
            relevant_ids_for_this_query.append(current_id)

    # Store this query's info: its text and which passage IDs are truly relevant to it.
    queries_data.append({
        "query_id": query_id,
        "query_text": query_text,
        "relevant_passage_ids": relevant_ids_for_this_query
    })

# Sanity checks: let's see how much data we ended up with.
print("Total unique passages in corpus:", len(corpus))
print("Total queries:", len(queries_data))
print("\nExample query entry:")
print(queries_data[0])
print("\nExample passage text (id 0):")
print(corpus[0])

Total unique passages in corpus: 24772
Total queries: 3000

Example query entry:
{'query_id': 19699, 'query_text': 'what is rba', 'relevant_passage_ids': [5]}

Example passage text (id 0):
Since 2007, the RBA's outstanding reputation has been affected by the 'Securency' or NPA scandal. These RBA subsidiaries were involved in bribing overseas officials so that Australia might win lucrative note-printing contracts. The assets of the bank include the gold and foreign exchange reserves of Australia, which is estimated to have a net worth of A$101 billion. Nearly 94% of the RBA's employees work at its headquarters in Sydney, New South Wales and at the Business Resumption Site.


In [13]:
import re
from rank_bm25 import BM25Okapi

def simple_tokenize(text):
    """
    Converts raw text into a list of clean, lowercase word tokens.
    Example: "What is BM25?" -> ["what", "is", "bm25"]
    """
    # .lower() converts all characters to lowercase, so "Diabetes" and "diabetes"
    # are treated as the exact same token.
    text = text.lower()

    # re.findall(r"\b\w+\b", text) uses a "regular expression" (a pattern-matching
    # tool for text) to pull out only word characters (letters/digits), splitting
    # on anything else (spaces, punctuation, symbols).
    # \b = word boundary, \w+ = one or more word characters.
    # So "what's up?" becomes ["what", "s", "up"] — punctuation is discarded.
    tokens = re.findall(r"\b\w+\b", text)

    return tokens

# corpus is {passage_id: passage_text} from Step 6.
# We need two aligned lists: one of passage_ids (to know which score belongs to
# which passage), and one of tokenized texts (what BM25 actually operates on).
passage_ids = list(corpus.keys())
tokenized_corpus = [simple_tokenize(corpus[pid]) for pid in passage_ids]

# Build the BM25 index. This is a ONE-TIME setup step — it precomputes term
# frequencies, document lengths, and IDF values across our whole corpus.
# This might take a few seconds depending on corpus size.
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index built successfully.")
print("Number of documents indexed:", len(tokenized_corpus))

BM25 index built successfully.
Number of documents indexed: 24772


In [2]:
def bm25_search(query_text, top_k=5):
    tokenized_query = simple_tokenize(query_text)
    scores = bm25.get_scores(tokenized_query)
    id_score_pairs = list(zip(passage_ids, scores))
    ranked = sorted(id_score_pairs, key=lambda pair: pair[1], reverse=True)
    return ranked[:top_k]

In [3]:
def dense_search(query_text, top_k=5):
    query_embedding = model.encode(
        [query_text], convert_to_numpy=True, normalize_embeddings=True
    )
    scores, indices = index.search(query_embedding, top_k)
    scores = scores[0]
    indices = indices[0]
    results = []
    for position, score in zip(indices, scores):
        pid = passage_ids[position]
        results.append((pid, score))
    return results

In [4]:
import numpy as np

def normalize_scores(scores):
    scores = np.array(scores)
    min_score = scores.min()
    max_score = scores.max()
    if max_score - min_score == 0:
        return np.zeros_like(scores)
    return (scores - min_score) / (max_score - min_score)

def hybrid_search(query_text, top_k=5, alpha=0.5):
    tokenized_query = simple_tokenize(query_text)
    bm25_scores_raw = bm25.get_scores(tokenized_query)
    query_embedding = model.encode(
        [query_text], convert_to_numpy=True, normalize_embeddings=True
    )
    dense_scores_raw, dense_indices = index.search(query_embedding, len(passage_ids))
    dense_scores_raw = dense_scores_raw[0]
    dense_indices = dense_indices[0]
    dense_scores_aligned = np.zeros(len(passage_ids))
    for position, score in zip(dense_indices, dense_scores_raw):
        dense_scores_aligned[position] = score
    bm25_norm = normalize_scores(bm25_scores_raw)
    dense_norm = normalize_scores(dense_scores_aligned)
    hybrid_scores = alpha * bm25_norm + (1 - alpha) * dense_norm
    id_score_pairs = list(zip(passage_ids, hybrid_scores))
    ranked = sorted(id_score_pairs, key=lambda pair: pair[1], reverse=True)
    return ranked[:top_k]

In [5]:
import time

def precision_at_k(retrieved_ids, relevant_ids, k):
    top_k_ids = retrieved_ids[:k]
    num_relevant_in_top_k = len(set(top_k_ids) & set(relevant_ids))
    return num_relevant_in_top_k / k

def recall_at_k(retrieved_ids, relevant_ids, k):
    if len(relevant_ids) == 0:
        return 0.0
    top_k_ids = retrieved_ids[:k]
    num_relevant_in_top_k = len(set(top_k_ids) & set(relevant_ids))
    return num_relevant_in_top_k / len(relevant_ids)

def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, pid in enumerate(retrieved_ids, start=1):
        if pid in relevant_ids:
            return 1.0 / rank
    return 0.0

In [6]:
def evaluate_method(search_function, eval_queries, k=5, **kwargs):
    precisions, recalls, reciprocal_ranks, times = [], [], [], []
    for q in eval_queries:
        query_text = q["query_text"]
        relevant_ids = q["relevant_passage_ids"]
        if len(relevant_ids) == 0:
            continue
        start_time = time.time()
        results = search_function(query_text, top_k=k, **kwargs)
        elapsed = time.time() - start_time
        times.append(elapsed)
        retrieved_ids = [pid for pid, score in results]
        precisions.append(precision_at_k(retrieved_ids, relevant_ids, k))
        recalls.append(recall_at_k(retrieved_ids, relevant_ids, k))
        reciprocal_ranks.append(reciprocal_rank(retrieved_ids, relevant_ids))
    return {
        "Precision@{}".format(k): sum(precisions)/len(precisions),
        "Recall@{}".format(k): sum(recalls)/len(recalls),
        "MRR": sum(reciprocal_ranks)/len(reciprocal_ranks),
        "Avg Time (sec)": sum(times)/len(times),
        "Num Queries Evaluated": len(precisions)
    }

In [14]:
import pandas as pd

EVAL_SAMPLE_SIZE = 200
eval_queries = queries_data[:EVAL_SAMPLE_SIZE]

bm25_results = evaluate_method(bm25_search, eval_queries, k=5)
dense_results = evaluate_method(dense_search, eval_queries, k=5)
hybrid_results = evaluate_method(hybrid_search, eval_queries, k=5, alpha=0.5)

comparison_df = pd.DataFrame({
    "BM25": bm25_results,
    "Dense": dense_results,
    "Hybrid (alpha=0.5)": hybrid_results
})
print(comparison_df)

                             BM25       Dense  Hybrid (alpha=0.5)
Precision@5              0.142857    0.179592            0.176531
Recall@5                 0.631803    0.810374            0.795068
MRR                      0.397194    0.519558            0.491071
Avg Time (sec)           0.105489    0.025352            0.225369
Num Queries Evaluated  196.000000  196.000000          196.000000


In [15]:
alpha_values = [0.0, 0.2, 0.3, 0.5, 0.7, 0.8, 1.0]
alpha_results = {}
for alpha in alpha_values:
    print(f"Evaluating Hybrid with alpha={alpha}...")
    alpha_results[f"alpha={alpha}"] = evaluate_method(hybrid_search, eval_queries, k=5, alpha=alpha)

alpha_comparison_df = pd.DataFrame(alpha_results)
print(alpha_comparison_df)

Evaluating Hybrid with alpha=0.0...
Evaluating Hybrid with alpha=0.2...
Evaluating Hybrid with alpha=0.3...
Evaluating Hybrid with alpha=0.5...
Evaluating Hybrid with alpha=0.7...
Evaluating Hybrid with alpha=0.8...
Evaluating Hybrid with alpha=1.0...
                        alpha=0.0   alpha=0.2   alpha=0.3   alpha=0.5  \
Precision@5              0.179592    0.191837    0.190816    0.176531   
Recall@5                 0.810374    0.869048    0.863946    0.795068   
MRR                      0.520408    0.536820    0.546769    0.491071   
Avg Time (sec)           0.216557    0.203579    0.201599    0.203895   
Num Queries Evaluated  196.000000  196.000000  196.000000  196.000000   

                        alpha=0.7   alpha=0.8   alpha=1.0  
Precision@5              0.163265    0.158163    0.142857  
Recall@5                 0.728741    0.700680    0.631803  
MRR                      0.438180    0.427381    0.397194  
Avg Time (sec)           0.200594    0.201869    0.199960  
Num Queri

In [12]:
def bm25_search(query_text, top_k=5):
    """
    Given a raw query string, returns the top_k most relevant passages
    according to BM25, along with their scores.
    """
    # Tokenize the query using the EXACT SAME tokenizer we used on the corpus.
    # If we used a different tokenizer here, words might not match correctly
    # (e.g., if corpus tokens are lowercase but query tokens aren't).
    tokenized_query = simple_tokenize(query_text)

    # Get a BM25 score for every passage in the corpus, in the same order
    # as passage_ids / tokenized_corpus.
    scores = bm25.get_scores(tokenized_query)

    # Pair up each passage_id with its score.
    # zip(passage_ids, scores) creates pairs like (id_0, score_0), (id_1, score_1), ...
    id_score_pairs = list(zip(passage_ids, scores))

    # Sort these pairs by score, highest first.
    # key=lambda pair: pair[1] tells sorted() to look at the SECOND item
    # in each pair (the score) when deciding order.
    # reverse=True means descending order (highest score first).
    ranked = sorted(id_score_pairs, key=lambda pair: pair[1], reverse=True)

    # Take only the top_k results.
    top_results = ranked[:top_k]

    return top_results


# Let's test it with a real example query from our own dataset,
# so we can sanity-check whether BM25 finds something sensible.
test_query = queries_data[0]["query_text"]
print("Query:", test_query)
print()

results = bm25_search(test_query, top_k=5)

for rank, (pid, score) in enumerate(results, start=1):
    print(f"Rank {rank} | Score: {score:.4f} | Passage ID: {pid}")
    print("Text:", corpus[pid][:200], "...")  # print first 200 characters only
    print()

Query: what is rba

Rank 1 | Score: 17.1712 | Passage ID: 8
Text: A rebuildable atomizer (RBA), often referred to as simply a “rebuildable,” is just a special type of atomizer used in the Vape Pen and Mod Industry that connects to a personal vaporizer. 1 The bottom  ...

Rank 2 | Score: 16.7254 | Passage ID: 3
Text: The inner workings of a rebuildable atomizer are surprisingly simple. The coil inside the RBA is made of some type of resistance wire, normally Kanthal or nichrome. When a current is applied to the co ...

Rank 3 | Score: 16.5910 | Passage ID: 4
Text: Results-Based Accountability® (also known as RBA) is a disciplined way of thinking and taking action that communities can use to improve the lives of children, youth, families, adults and the communit ...

Rank 4 | Score: 15.3229 | Passage ID: 5
Text: Results-Based Accountability® (also known as RBA) is a disciplined way of thinking and taking action that communities can use to improve the lives of children, youth, families, a

In [7]:
from sentence_transformers import SentenceTransformer

# Load the pretrained model. First time this runs, it downloads the model
# (~80MB) from Hugging Face and caches it locally — after that, loading
# is instant since it just reads from your local cache.
model = SentenceTransformer("all-MiniLM-L6-v2")

# We need our passage texts as a plain list, in the SAME order as passage_ids,
# so that embedding[i] corresponds to passage_ids[i] — this alignment matters
# for everything downstream (FAISS search, hybrid combination, evaluation).
passage_texts_list = [corpus[pid] for pid in passage_ids]

# Encode all passages into embeddings.
# show_progress_bar=True displays a progress bar, since this can take a
# little while depending on corpus size and your CPU.
# convert_to_numpy=True returns a NumPy array instead of a PyTorch tensor,
# which is the format FAISS (Step 9) expects.
passage_embeddings = model.encode(
    passage_texts_list,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", passage_embeddings.shape)

c:\Users\srinidhi\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Batches:   0%|          | 0/775 [00:00<?, ?it/s]

Embeddings shape: (24772, 384)


In [8]:
import sys
!{sys.executable} -m pip install tf-keras


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import faiss
import numpy as np

# Re-encode passages WITH normalization this time, since we didn't normalize
# in Step 8. normalize_embeddings=True makes every vector have length 1,
# which is required for inner product to behave like cosine similarity.
passage_embeddings = model.encode(
    passage_texts_list,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Get the embedding dimension (384 for this model) directly from the data,
# rather than hardcoding the number, so our code stays correct even if we
# ever switch to a different embedding model later.
embedding_dim = passage_embeddings.shape[1]

# Create a FAISS index: "Flat" = exact search, "IP" = inner product similarity.
index = faiss.IndexFlatIP(embedding_dim)

# Add all our passage vectors into the index. This is the "build" step —
# after this, the index is ready to be searched against.
index.add(passage_embeddings)

print("Number of vectors in FAISS index:", index.ntotal)

Batches:   0%|          | 0/775 [00:00<?, ?it/s]

Number of vectors in FAISS index: 24772


In [12]:
import pickle

# Save the FAISS index itself to a file on disk.
faiss.write_index(index, "faiss_index.bin")

# Save everything else we'll need later (corpus, passage_ids, queries_data).
with open("saved_data.pkl", "wb") as f:
    pickle.dump({
        "corpus": corpus,
        "passage_ids": passage_ids,
        "queries_data": queries_data,
    }, f)

print("Saved FAISS index and supporting data to disk.")

Saved FAISS index and supporting data to disk.


In [8]:
import faiss
import pickle
from sentence_transformers import SentenceTransformer

index = faiss.read_index("faiss_index.bin")

with open("saved_data.pkl", "rb") as f:
    saved = pickle.load(f)

corpus = saved["corpus"]
passage_ids = saved["passage_ids"]
queries_data = saved["queries_data"]

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Reloaded everything from disk successfully.")
print("FAISS vectors:", index.ntotal)
print("Corpus size:", len(corpus))

c:\Users\srinidhi\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):



Reloaded everything from disk successfully.
FAISS vectors: 24772
Corpus size: 24772


In [2]:
def dense_search(query_text, top_k=5):
    """
    Given a raw query string, returns the top_k most relevant passages
    according to Dense Retrieval (embedding similarity via FAISS).
    """
    # Embed the query using the SAME model, with the SAME normalization
    # setting we used for the corpus — consistency is essential, otherwise
    # similarity scores would be meaningless.
    query_embedding = model.encode(
        [query_text],  # .encode() expects a list, even for a single query
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Search the FAISS index. Returns:
    # - scores: a 2D array of similarity scores, shape (1, top_k)
    # - indices: a 2D array of matching positions, shape (1, top_k)
    # We use [0] on both since we only searched ONE query (batch size 1).
    scores, indices = index.search(query_embedding, top_k)
    scores = scores[0]
    indices = indices[0]

    # Convert FAISS's internal positions back into our real passage_ids,
    # and pair each with its similarity score.
    results = []
    for position, score in zip(indices, scores):
        pid = passage_ids[position]
        results.append((pid, score))

    return results


# Test with the same query we used for BM25, so we can directly compare later.
test_query = queries_data[0]["query_text"]
print("Query:", test_query)
print()

results = dense_search(test_query, top_k=5)

for rank, (pid, score) in enumerate(results, start=1):
    print(f"Rank {rank} | Score: {score:.4f} | Passage ID: {pid}")
    print("Text:", corpus[pid][:200], "...")
    print()

Query: what is rba

Rank 1 | Score: 0.8350 | Passage ID: 9
Text: Get To Know Us. RBA is a digital and technology consultancy with roots in strategy, design and technology. Our team of specialists help progressive companies deliver modern digital experiences backed  ...

Rank 2 | Score: 0.6461 | Passage ID: 2
Text: RBA Recognized with the 2014 Microsoft US Regional Partner of the ... by PR Newswire. Contract Awarded for supply and support the. Securitisations System used for risk management and analysis.  ...

Rank 3 | Score: 0.6370 | Passage ID: 7
Text: vs. NetIQ Identity Manager. Risk-based authentication (RBA) is a method of applying varying levels of stringency to authentication processes based on the likelihood that access to a given system could ...

Rank 4 | Score: 0.6332 | Passage ID: 6
Text: RBA uses a data-driven, decision-making process to help communities and organizations get beyond talking about problems to taking action to solve problems. It is a simple, common sense fram

In [5]:
import numpy as np

def normalize_scores(scores):
    """
    Rescales a list/array of scores to the 0-1 range using min-max normalization.
    """
    scores = np.array(scores)
    min_score = scores.min()
    max_score = scores.max()

    if max_score - min_score == 0:
        return np.zeros_like(scores)

    return (scores - min_score) / (max_score - min_score)


def hybrid_search(query_text, top_k=5, alpha=0.5):
    """
    Combines BM25 and Dense Retrieval scores into a single ranked list.
    alpha controls the weighting: alpha=1 -> pure BM25, alpha=0 -> pure Dense.
    """
    tokenized_query = simple_tokenize(query_text)
    bm25_scores_raw = bm25.get_scores(tokenized_query)

    query_embedding = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    dense_scores_raw, dense_indices = index.search(query_embedding, len(passage_ids))
    dense_scores_raw = dense_scores_raw[0]
    dense_indices = dense_indices[0]

    dense_scores_aligned = np.zeros(len(passage_ids))
    for position, score in zip(dense_indices, dense_scores_raw):
        dense_scores_aligned[position] = score

    bm25_norm = normalize_scores(bm25_scores_raw)
    dense_norm = normalize_scores(dense_scores_aligned)

    hybrid_scores = alpha * bm25_norm + (1 - alpha) * dense_norm

    id_score_pairs = list(zip(passage_ids, hybrid_scores))
    ranked = sorted(id_score_pairs, key=lambda pair: pair[1], reverse=True)
    return ranked[:top_k]


test_query = queries_data[0]["query_text"]
print("Query:", test_query)
print()

results = hybrid_search(test_query, top_k=5, alpha=0.5)

for rank, (pid, score) in enumerate(results, start=1):
    print(f"Rank {rank} | Hybrid Score: {score:.4f} | Passage ID: {pid}")
    print("Text:", corpus[pid][:200], "...")
    print()

Query: what is rba

Rank 1 | Hybrid Score: 0.8988 | Passage ID: 8
Text: A rebuildable atomizer (RBA), often referred to as simply a “rebuildable,” is just a special type of atomizer used in the Vape Pen and Mod Industry that connects to a personal vaporizer. 1 The bottom  ...

Rank 2 | Hybrid Score: 0.8802 | Passage ID: 9
Text: Get To Know Us. RBA is a digital and technology consultancy with roots in strategy, design and technology. Our team of specialists help progressive companies deliver modern digital experiences backed  ...

Rank 3 | Hybrid Score: 0.8763 | Passage ID: 4
Text: Results-Based Accountability® (also known as RBA) is a disciplined way of thinking and taking action that communities can use to improve the lives of children, youth, families, adults and the communit ...

Rank 4 | Hybrid Score: 0.8417 | Passage ID: 6
Text: RBA uses a data-driven, decision-making process to help communities and organizations get beyond talking about problems to taking action to solve problems

In [7]:
import time

def precision_at_k(retrieved_ids, relevant_ids, k):
    """Fraction of top-k retrieved items that are actually relevant."""
    top_k_ids = retrieved_ids[:k]
    num_relevant_in_top_k = len(set(top_k_ids) & set(relevant_ids))
    return num_relevant_in_top_k / k


def recall_at_k(retrieved_ids, relevant_ids, k):
    """Fraction of all relevant items that were found in the top-k."""
    if len(relevant_ids) == 0:
        return 0.0  # can't compute recall if there's nothing relevant to find
    top_k_ids = retrieved_ids[:k]
    num_relevant_in_top_k = len(set(top_k_ids) & set(relevant_ids))
    return num_relevant_in_top_k / len(relevant_ids)


def reciprocal_rank(retrieved_ids, relevant_ids):
    """1 / rank of the first relevant result found; 0 if none found."""
    for rank, pid in enumerate(retrieved_ids, start=1):
        if pid in relevant_ids:
            return 1.0 / rank
    return 0.0

In [14]:
import time

def evaluate_method(search_function, eval_queries, k=5, **kwargs):
    """
    Runs a given search function across all eval_queries, computes average
    Precision@k, Recall@k, MRR, and average retrieval time.
    """
    precisions = []
    recalls = []
    reciprocal_ranks = []
    times = []

    for q in eval_queries:
        query_text = q["query_text"]
        relevant_ids = q["relevant_passage_ids"]

        # Skip queries with no ground-truth relevant passages at all —
        # Precision/Recall/MRR are undefined/meaningless for these.
        if len(relevant_ids) == 0:
            continue

        # Time how long this one search call takes.
        start_time = time.time()
        results = search_function(query_text, top_k=k, **kwargs)
        elapsed = time.time() - start_time
        times.append(elapsed)

        # results is a list of (passage_id, score) pairs — we only need the IDs,
        # in ranked order, for our metric functions.
        retrieved_ids = [pid for pid, score in results]

        precisions.append(precision_at_k(retrieved_ids, relevant_ids, k))
        recalls.append(recall_at_k(retrieved_ids, relevant_ids, k))
        reciprocal_ranks.append(reciprocal_rank(retrieved_ids, relevant_ids))

    # Average each metric across all evaluated queries.
    avg_precision = sum(precisions) / len(precisions)
    avg_recall = sum(recalls) / len(recalls)
    avg_mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
    avg_time = sum(times) / len(times)

    return {
        "Precision@{}".format(k): avg_precision,
        "Recall@{}".format(k): avg_recall,
        "MRR": avg_mrr,
        "Avg Time (sec)": avg_time,
        "Num Queries Evaluated": len(precisions)
    }


# Use a smaller sample for evaluation to keep runtime reasonable.
# Dense/Hybrid involve a neural network call per query, so 200 queries
# is a reasonable balance between speed and statistical reliability.
EVAL_SAMPLE_SIZE = 200
eval_queries = queries_data[:EVAL_SAMPLE_SIZE]

print("Evaluating BM25...")
bm25_results = evaluate_method(bm25_search, eval_queries, k=5)

print("Evaluating Dense Retrieval...")
dense_results = evaluate_method(dense_search, eval_queries, k=5)

print("Evaluating Hybrid Search (alpha=0.5)...")
hybrid_results = evaluate_method(hybrid_search, eval_queries, k=5, alpha=0.5)

# Print a clean comparison table.
import pandas as pd

comparison_df = pd.DataFrame({
    "BM25": bm25_results,
    "Dense": dense_results,
    "Hybrid (alpha=0.5)": hybrid_results
})

print(comparison_df)

Evaluating BM25...
Evaluating Dense Retrieval...
Evaluating Hybrid Search (alpha=0.5)...
                             BM25       Dense  Hybrid (alpha=0.5)
Precision@5              0.142857    0.179592            0.176531
Recall@5                 0.631803    0.810374            0.795068
MRR                      0.397194    0.519558            0.491071
Avg Time (sec)           0.104885    0.024642            0.190821
Num Queries Evaluated  196.000000  196.000000          196.000000


In [13]:
print("bm25_search defined:", "bm25_search" in dir())
print("dense_search defined:", "dense_search" in dir())
print("hybrid_search defined:", "hybrid_search" in dir())
print("bm25 defined:", "bm25" in dir())
print("simple_tokenize defined:", "simple_tokenize" in dir())

bm25_search defined: True
dense_search defined: True
hybrid_search defined: True
bm25 defined: True
simple_tokenize defined: True


In [15]:
alpha_values = [0.0, 0.2, 0.3, 0.5, 0.7, 0.8, 1.0]

alpha_results = {}

for alpha in alpha_values:
    print(f"Evaluating Hybrid with alpha={alpha}...")
    result = evaluate_method(hybrid_search, eval_queries, k=5, alpha=alpha)
    alpha_results[f"alpha={alpha}"] = result

alpha_comparison_df = pd.DataFrame(alpha_results)
print(alpha_comparison_df)

Evaluating Hybrid with alpha=0.0...
Evaluating Hybrid with alpha=0.2...
Evaluating Hybrid with alpha=0.3...
Evaluating Hybrid with alpha=0.5...
Evaluating Hybrid with alpha=0.7...
Evaluating Hybrid with alpha=0.8...
Evaluating Hybrid with alpha=1.0...
                        alpha=0.0   alpha=0.2   alpha=0.3   alpha=0.5  \
Precision@5              0.179592    0.191837    0.190816    0.176531   
Recall@5                 0.810374    0.869048    0.863946    0.795068   
MRR                      0.520408    0.536820    0.546769    0.491071   
Avg Time (sec)           0.197614    0.201368    0.194026    0.189818   
Num Queries Evaluated  196.000000  196.000000  196.000000  196.000000   

                        alpha=0.7   alpha=0.8   alpha=1.0  
Precision@5              0.163265    0.158163    0.142857  
Recall@5                 0.728741    0.700680    0.631803  
MRR                      0.438180    0.427381    0.397194  
Avg Time (sec)           0.188879    0.509145    0.302317  
Num Queri

In [16]:
# Save your main 3-way comparison (BM25 vs Dense vs Hybrid at alpha=0.3, your best value).
comparison_df.to_csv("comparison_results.csv")

# Save your full alpha sweep results too.
alpha_comparison_df.to_csv("alpha_sweep_results.csv")

print("Saved evaluation results to CSV.")

Saved evaluation results to CSV.
